# v8 - boundary stress test of raw GLORYS under `from_nemo`

Raw GLORYS only. `FieldSet.from_nemo` (which negates W itself), `ScipyParticle`, and exactly two
kernels: the stock `AdvectionRK4_3D` and v7's `CheckError`.

## Ground rules, and why they matter here

**Nothing is allowed to touch the physics.** No reflection, no clamping, no depth limiter, no
`Diagnose` kernel, no velocity cap. Every boundary quantity in this notebook is computed
*after the fact* from the written trajectory and the mesh, so the numbers are the model's
behaviour rather than a kernel's. `CheckError` deletes a particle whose state is an error and
records the `StatusCode` that killed it - a status code is measurement, not interference: it
changes no position and no velocity, and without it the losses cannot be told apart.

That is the whole point of running this on **raw GLORYS**: v7 established that GLORYS is the
one field with a broken surface boundary condition (`W(0) != 0`) and an unclosed volume budget
(`rms |D| = 2.1e-07 m/s`), and the only one of the three to put particles through the `w = 0`
seabed face. v7 seeded two plausible release regions. This notebook seeds the **boundaries
themselves** and asks where the field actually fails.

## Eleven seed classes, each aimed at one boundary

| class | placement | what it probes |
|---|---|---|
| `surface` | `z = 0.0` exactly | the top face, where GLORYS' `W(0) != 0` |
| `subsurf_1cm` | `z = 0.01 m` | just inside the surface cell |
| `floor_face` | `z = gdepw_0[mbathy]` | the field's own zero-`w` face |
| `partial_gap` | between `H` and `gdepw_0[mbathy]` | the partial-cell gap: below the true seabed, still open water to the field |
| `deepest_cell` | middle of the deepest wet cell | the last cell with a T-point |
| `coast` | wet cell with a land neighbour | lateral land masking |
| `rim_north/south/east/west` | within 2 cells of each domain edge | the four open lateral boundaries |
| `interior_deep` | 500 m in columns deeper than 3000 m | **control** - should not fail |

`interior_deep` is the control: if it fails, the problem is not a boundary.

## Predictions, recorded before the run

1. `surface` should be the worst class. It sits exactly on the face where GLORYS' `W(0)` is
   nonzero, and with the sign correct a positive `W(0)` is upward motion at `z = 0`, which is
   `ErrorThroughSurface` with no clamping to absorb it. v7 saw zero surface losses from a
   0-10 m release; seeding *at* `z = 0` is the sharper test.
2. `rim_*` should lose particles steadily to `ErrorOutOfBounds` and this is **not a defect** -
   an open lateral boundary is supposed to let water out. It is reported as a rate, not a bug.
3. `partial_gap` should survive. v4 assumed the field's floor, not the true seabed, is the
   real barrier; particles started inside the gap should therefore advect normally rather than
   error immediately.
4. `interior_deep` should record zero deaths and no implied speed above ~2 m/s.
5. **No particle should ever exceed ~3 m/s implied speed.** GLORYS is a 1/12-degree daily-mean
   product; the North Brazil Current tops out near 1.5-2 m/s. Anything faster is an
   interpolation artefact, and "flying away" means exactly that.

Section 6 reads all five back. They are not edited afterwards.

## A Parcels gotcha this notebook has to work around

`Kernel.__init__` injects `math`, `ParcelsRandom`, `rng`, `random` and `StatusCode` into the
**outermost frame's globals** (`parcels/kernel.py:219-225`). A notebook-level variable named
`rng` is silently replaced by Parcels' RNG module the first time a kernel is built, after which
`rng.uniform(a, b, n)` raises `TypeError: uniform() takes 2 positional arguments but 3 were
given`. The seeding below therefore uses `seed_rng`, never `rng`, so the seed cell stays
re-runnable after a run.

## 1. Configuration

In [ ]:
n_per_class        = 150
run_time_days      = 20            # ScipyParticle costs ~2 ms per particle-step; sized for ~1.5 h
time_step_minutes  = 20
out_put_step_hours = 3             # finer than v7: boundary failures are fast

ref_date, offset = "1993-01-01", 1
rdm_seed = 110987
MONTHS = [1, 2, 3]

Hgr, Zgr = "Hgr_cmesh.nc", "Zgr_cmesh2.nc"

def mf(pat):
    return [pat.format(y=1993, m=m) for m in MONTHS]

# raw GLORYS, untouched
GLORYS = dict(U=mf("U_{y}-{m:02d}.nc"), V=mf("V_{y}-{m:02d}.nc"), W=mf("W_{y}-{m:02d}.nc"))

# what counts as "flying away": GLORYS is a 1/12-deg daily mean; the NBC peaks near 1.5-2 m/s
SPEED_BINS = [1.0, 2.0, 3.0, 5.0, 10.0]

out_root    = "tracks_v8"
FORCE_RERUN = False

In [ ]:
import os, sys, shutil, contextlib, textwrap, time
import numpy as np
import xarray as xr
from datetime import timedelta
from scipy.spatial import cKDTree
import matplotlib as mpl
import matplotlib.pyplot as plt

import parcels
from parcels import (FieldSet, ParticleSet, ScipyParticle, Variable,
                     AdvectionRK4_3D, StatusCode)

@contextlib.contextmanager
def quiet():
    sys.stdout.flush()
    saved, devnull = os.dup(1), os.open(os.devnull, os.O_WRONLY)
    try:
        os.dup2(devnull, 1); yield
    finally:
        os.dup2(saved, 1); os.close(devnull); os.close(saved)

print("parcels", parcels.__version__)
miss = [p for v in ("U", "V", "W") for p in GLORYS[v] if not os.path.exists(p)]
print("GLORYS", "OK" if not miss else "MISSING: " + ", ".join(miss))

start_time = np.datetime64(ref_date) + np.timedelta64(offset, "D")
nout = int(run_time_days * 24 / out_put_step_hours) + 1
print(f"{start_time} -> {start_time + np.timedelta64(run_time_days,'D')}"
      f"   dt={time_step_minutes} min   {nout} outputs")

# the StatusCode values the death statistics are keyed on
CODES = {getattr(StatusCode, n): n for n in
         ("Success", "Evaluate", "Repeat", "Delete", "StopExecution", "Error",
          "ErrorInterpolation", "ErrorOutOfBounds", "ErrorThroughSurface",
          "ErrorTimeExtrapolation") if hasattr(StatusCode, n)}
print("\nStatusCode:", ", ".join(f"{v}={k}" for k, v in sorted(CODES.items(), key=lambda x: x[1])))

## 2. Mesh, and the lookup tables the post-hoc diagnostics need

In [ ]:
zgr = xr.open_dataset(Zgr).squeeze()
hgr = xr.open_dataset(Hgr).squeeze()
mbathy  = zgr.mbathy.values.astype(int)
gdepw_0 = zgr.gdepw_0.squeeze().values.astype("f8")
e3t_ps  = zgr.e3t_ps.values.astype("f8")
lon2d, lat2d = zgr.nav_lon.values.astype("f8"), zgr.nav_lat.values.astype("f8")
nz, ny, nx = len(gdepw_0), *mbathy.shape

kb    = np.clip(mbathy - 1, 0, nz - 1)
thick = np.where(mbathy > 0, e3t_ps, 0.0)
H     = gdepw_0[kb] + thick                                        # true seabed
Gnom  = np.where(mbathy > 0, gdepw_0[np.clip(mbathy, 0, nz - 1)], 0.0)  # field's zero-w face
zbot  = gdepw_0[kb] + 0.5 * thick                                  # middle of deepest wet cell
ocean = mbathy > 0

# one KD-tree over EVERY column, wet and dry. Nearest-column lookups then give mbathy (so a
# particle sitting over land can be detected), H and Gnom, all without a Parcels field and
# therefore without any kernel.
gp   = np.column_stack([lon2d.ravel(), lat2d.ravel()])
tree = cKDTree(gp)
FLAT = dict(mbathy=mbathy.ravel(), H=H.ravel(), Gnom=Gnom.ravel())

def lookup(lo, la):
    '''nearest grid column -> (is_land, H, Gnom). NaN positions map to NaN.'''
    lo, la = np.asarray(lo, float), np.asarray(la, float)
    ok = np.isfinite(lo) & np.isfinite(la)
    out = {k: np.full(lo.shape, np.nan) for k in ("H", "Gnom")}
    land = np.zeros(lo.shape, bool)
    if ok.any():
        _, idx = tree.query(np.column_stack([lo[ok], la[ok]]))
        land[ok] = FLAT["mbathy"][idx] == 0
        for k in ("H", "Gnom"):
            out[k][ok] = FLAT[k][idx]
    return land, out["H"], out["Gnom"]

print(f"grid {ny} x {nx} x {nz}   {int(ocean.sum())} wet columns")
# gap statistics over WET columns only: `gap > 1` on a NaN-filled array yields a bool array
# with no NaN, so np.nanmean would divide by every cell in the grid, land included.
gap = (Gnom - H)[ocean]
print(f"partial-cell gap Gnom - H:  median {np.median(gap):.2f} m   mean {gap.mean():.2f} m   "
      f"> 1 m in {(gap > 1).mean() * 100:.0f}% of the {ocean.sum()} wet columns")

## 3. Seeds - eleven boundary classes

`seed_rng`, not `rng`, for the reason in the header. Every class is drawn from wet columns
only; nothing is seeded inside land.

In [ ]:
seed_rng = np.random.default_rng(rdm_seed)

def pick(mask, n):
    jj, ii = np.where(mask)
    if len(jj) == 0: return np.array([], int), np.array([], int)
    s = seed_rng.choice(len(jj), size=n, replace=len(jj) < n)
    return jj[s], ii[s]

interior = np.zeros_like(ocean); interior[2:-2, 2:-2] = True

# coast: wet, with at least one dry 4-neighbour
land_nb = np.zeros_like(ocean)
for sh, ax in ((1, 0), (-1, 0), (1, 1), (-1, 1)):
    land_nb |= np.roll(~ocean, sh, axis=ax)
coast = ocean & land_nb & interior

rim = {}
rim["rim_north"] = ocean & (np.arange(ny)[:, None] >= ny - 3)
rim["rim_south"] = ocean & (np.arange(ny)[:, None] <= 2)
rim["rim_east"]  = ocean & (np.arange(nx)[None, :] >= nx - 3)
rim["rim_west"]  = ocean & (np.arange(nx)[None, :] <= 2)

CLASSES = {}
def add(name, mask, zfun):
    j, i = pick(mask, n_per_class)
    if len(j) == 0:
        print(f"  {name:<14} EMPTY - no cells match"); return
    CLASSES[name] = (lon2d[j, i], lat2d[j, i], zfun(j, i))

deep = ocean & interior & (H > 3000)
mid  = lambda j, i: 0.5 * H[j, i]

add("surface",       ocean & interior,                      lambda j, i: np.zeros(len(j)))
add("subsurf_1cm",   ocean & interior,                      lambda j, i: np.full(len(j), 0.01))
add("floor_face",    ocean & interior,                      lambda j, i: Gnom[j, i])
add("partial_gap",   ocean & interior & (Gnom - H > 1.0),
    lambda j, i: H[j, i] + seed_rng.uniform(0.05, 0.95, len(j)) * (Gnom[j, i] - H[j, i]))
add("deepest_cell",  ocean & interior,                      lambda j, i: zbot[j, i])
add("coast",         coast,                                 mid)
for nm, m in rim.items():
    add(nm, m, mid)
add("interior_deep", deep,                                  lambda j, i: np.full(len(j), 500.0))

NAMES = list(CLASSES)
start_lon = np.concatenate([CLASSES[k][0] for k in NAMES])
start_lat = np.concatenate([CLASSES[k][1] for k in NAMES])
start_dep = np.concatenate([CLASSES[k][2] for k in NAMES])
start_cls = np.concatenate([np.full(len(CLASSES[nm][0]), k) for k, nm in enumerate(NAMES)])
npart = len(start_lon)

print(f"{'class':<15}{'n':>5}{'z min':>9}{'z max':>9}{'H min':>9}{'H max':>9}")
for k, nm in enumerate(NAMES):
    lo, la, z = CLASSES[nm]
    _, hh, _ = lookup(lo, la)
    print(f"{nm:<15}{len(lo):>5}{z.min():>9.2f}{z.max():>9.2f}{np.nanmin(hh):>9.1f}{np.nanmax(hh):>9.1f}")
print(f"\n{npart} particles, {run_time_days} days, dt {time_step_minutes} min, ScipyParticle")
print(f"estimated cost: {npart * run_time_days * 24 * 60 / time_step_minutes * 1.96e-3 / 60:.0f} min"
      f"   (measured 1.96 ms per particle-step for Scipy mode)")

## 4. Kernels - the stock advection, and `CheckError`. Nothing else

`AdvectionRK4_3D` is imported from Parcels, not retyped, so it cannot drift. `CheckError` is
v7's: it records the status code that killed a particle and deletes it. No other kernel runs,
so nothing in this notebook can move a particle except the advection itself.

In [ ]:
def CheckError(particle, fieldset, time):  # pragma: no cover
    '''Records why a particle died before deleting it. Writes a status code and nothing else -
    no position, depth or velocity is touched, so the physics is untouched. Deleting rather
    than resetting the state means a failure is counted, not silently survived.'''
    if particle.state >= 50:
        particle.died = particle.state
        particle.delete()


class BoundaryParticle(ScipyParticle):
    cls  = Variable("cls",  initial=0, dtype=np.int32, to_write="once")
    died = Variable("died", initial=0, dtype=np.int32)

print("kernels:", [AdvectionRK4_3D.__name__, CheckError.__name__])
print("particle:", BoundaryParticle.__mro__[1].__name__)
import inspect
print("\nAdvectionRK4_3D is Parcels' own, unmodified:")
print(textwrap.indent("".join(inspect.getsource(AdvectionRK4_3D).splitlines(keepends=True)[:4]), "    "))

## 5. Run

In [ ]:
def run():
    out = os.path.join(out_root, "glorys_boundaries.zarr")
    if os.path.exists(out) and not FORCE_RERUN:
        print(f"  reusing {out}"); return out
    if os.path.exists(out): shutil.rmtree(out)
    os.makedirs(out_root, exist_ok=True)
    fn = {v: {"data": GLORYS[v], "lon": Hgr, "lat": Hgr, "depth": GLORYS["W"][0]}
          for v in ("U", "V", "W")}
    fs = FieldSet.from_nemo(fn, {"U": "vozocrtx", "V": "vomecrty", "W": "vovecrtz"},
                            {v: {"lon": "glamf", "lat": "gphif", "depth": "depthw",
                                 "time": "time_counter"} for v in ("U", "V", "W")},
                            deferred_load=True, allow_time_extrapolation=False)
    ps = ParticleSet(fs, BoundaryParticle, lon=start_lon, lat=start_lat, depth=start_dep,
                     time=np.datetime64(start_time), cls=start_cls)
    pf = ps.ParticleFile(name=out, outputdt=timedelta(hours=out_put_step_hours),
                         chunks=(npart, nout + 2))
    t0 = time.time()
    with quiet():
        ps.execute([AdvectionRK4_3D, CheckError],
                   runtime=timedelta(days=run_time_days),
                   dt=timedelta(minutes=time_step_minutes), output_file=pf)
    print(f"  {time.time() - t0:.0f} s")
    return out

path = run()
d = xr.open_zarr(path).compute()
LON, LAT, Z = d.lon.values, d.lat.values, d.z.values
CLS  = d.cls.values
DIED = np.nanmax(np.nan_to_num(d.died.values), axis=1).astype(int)
NLIVE = np.isfinite(LAT).sum(1)
full  = NLIVE.max()
tdays = np.arange(LAT.shape[1]) * out_put_step_hours / 24.0
print(f"\nwritten {LAT.shape[0]} particles x {LAT.shape[1]} obs; longest-lived = {full} obs")

## 6. Survival and cause of death, by boundary class

In [ ]:
print(f"{'class':<15}{'n':>4}{'survived':>10}{'died':>6}   " +
      "".join(f"{CODES[c].replace('Error',''):>14}" for c in sorted(CODES) if c >= 50))
tot = {}
for k, nm in enumerate(NAMES):
    s = CLS == k
    lost = s & (NLIVE < full)
    row = f"{nm:<15}{int(s.sum()):>4}{int((s & ~lost).sum()):>10}{int(lost.sum()):>6}   "
    for c in sorted(CODES):
        if c < 50: continue
        cnt = int((lost & (DIED == c)).sum())
        tot[c] = tot.get(c, 0) + cnt
        row += f"{cnt if cnt else '.':>14}"
    print(row)
print(f"{'TOTAL':<15}{npart:>4}{int((NLIVE >= full).sum()):>10}{int((NLIVE < full).sum()):>6}   "
      + "".join(f"{tot.get(c, 0) if tot.get(c, 0) else '.':>14}" for c in sorted(CODES) if c >= 50))

print(f"\n{'class':<15}{'time to death (days)':>24}")
print(f"{'':<15}{'min':>8}{'median':>8}{'max':>8}")
for k, nm in enumerate(NAMES):
    lost = (CLS == k) & (NLIVE < full)
    if not lost.any():
        print(f"{nm:<15}{'-':>8}{'-':>8}{'-':>8}"); continue
    td = NLIVE[lost] * out_put_step_hours / 24.0
    print(f"{nm:<15}{td.min():>8.2f}{np.median(td):>8.2f}{td.max():>8.2f}")

## 7. Boundary excursions, computed from the output only

No `Diagnose` kernel ran. These come from the written `z` and a nearest-column mesh lookup, so
they are measurements of what the model did.

`below_F` - past the `k = mbathy` face where the data itself holds `w = 0` - is the only
unambiguous violation. `below_H` is expected to be positive: the partial-cell gap is open water
as far as the field is concerned.

In [ ]:
land_hit, H_at, G_at = lookup(LON, LAT)
above_S = np.where(np.isfinite(Z), -Z, np.nan)
below_H = np.where(np.isfinite(Z), Z - H_at, np.nan)
below_F = np.where(np.isfinite(Z), Z - G_at, np.nan)

print(f"{'class':<15}{'above surf n':>13}{'max m':>9}{'below F n':>11}{'max m':>9}"
      f"{'below H n':>11}{'max m':>9}{'over land n':>13}")
for k, nm in enumerate(NAMES):
    s = CLS == k
    a, f, h = above_S[s], below_F[s], below_H[s]
    mx = lambda x: np.nanmax(x) if np.isfinite(x).any() else np.nan
    per = lambda x: int((np.nanmax(x, axis=1) > 1e-6).sum())
    print(f"{nm:<15}{per(a):>13}{max(mx(a), 0):>9.3f}{per(f):>11}{max(mx(f), 0):>9.3f}"
          f"{per(h):>11}{max(mx(h), 0):>9.3f}{int(land_hit[s].any(axis=1).sum()):>13}")

print(textwrap.dedent('''
    "over land n" counts particles that were written at a position whose nearest grid column is
    dry. A C-grid velocity field is zero on a closed face, so this should be empty; anything
    here is a particle that crossed a land mask.'''))

## 8. Particles that fly away

Implied speed between consecutive writes: great-circle distance divided by the output interval.
This is a lower bound on the speed the particle actually experienced - a particle that loops
between writes reads slower than it moved - so anything flagged here is real.

In [ ]:
R_EARTH = 6371000.0
la1, la2 = np.deg2rad(LAT[:, :-1]), np.deg2rad(LAT[:, 1:])
lo1, lo2 = np.deg2rad(LON[:, :-1]), np.deg2rad(LON[:, 1:])
hav = np.sin((la2 - la1) / 2) ** 2 + np.cos(la1) * np.cos(la2) * np.sin((lo2 - lo1) / 2) ** 2
step_m = 2 * R_EARTH * np.arcsin(np.sqrt(np.clip(hav, 0, 1)))
vh = step_m / (out_put_step_hours * 3600.0)                    # m/s, horizontal
vz = np.abs(np.diff(Z, axis=1)) / (out_put_step_hours * 3600.0)  # m/s, vertical

print(f"{'class':<15}{'max |vh|':>10}{'p99.9':>9}{'median':>9}{'max |vz|':>10}"
      + "".join(f"{'>' + str(b):>8}" for b in SPEED_BINS))
for k, nm in enumerate(NAMES):
    s = CLS == k
    a, b = vh[s], vz[s]
    g = np.isfinite(a)
    if not g.any():
        print(f"{nm:<15}{'-':>10}"); continue
    row = (f"{nm:<15}{np.nanmax(a):>10.3f}{np.nanpercentile(a[g], 99.9):>9.3f}"
           f"{np.nanmedian(a[g]):>9.3f}{np.nanmax(b) if np.isfinite(b).any() else np.nan:>10.4f}")
    for thr in SPEED_BINS:
        n_ = int((np.nanmax(a, axis=1) > thr).sum())
        row += f"{n_ if n_ else '.':>8}"
    print(row)
print("\ncolumns '>x' = particles whose implied horizontal speed exceeded x m/s at least once")
print(f"whole-run max implied horizontal speed: {np.nanmax(vh):.3f} m/s"
      f"   vertical: {np.nanmax(vz):.4f} m/s")

## 9. Figures

In [ ]:
SURF, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#dedcd6"
plt.rcParams.update({"font.size": 9, "axes.edgecolor": GRID, "axes.labelcolor": INK2,
    "xtick.color": INK2, "ytick.color": INK2, "axes.facecolor": SURF,
    "figure.facecolor": SURF, "axes.titlecolor": INK})

fig, axs = plt.subplots(1, 3, figsize=(16.5, 4.6))

# survival curve per class
ax = axs[0]
cmap = mpl.colormaps["turbo"](np.linspace(0.05, 0.95, len(NAMES)))
for k, nm in enumerate(NAMES):
    s = CLS == k
    alive = np.isfinite(LAT[s]).sum(axis=0) / max(int(s.sum()), 1) * 100
    ax.plot(tdays, alive, lw=1.5, color=cmap[k], label=nm)
ax.set_xlabel("days"); ax.set_ylabel("% still alive"); ax.set_ylim(-2, 103)
ax.set_title("survival by boundary class", loc="left", fontsize=10.5, pad=8)
ax.legend(frameon=False, fontsize=6.6, ncol=2)

# implied-speed distribution
ax = axs[1]
for k, nm in enumerate(NAMES):
    a = vh[CLS == k]; a = a[np.isfinite(a) & (a > 0)]
    if a.size < 10: continue
    q = np.linspace(0, 100, 200)
    ax.plot(np.percentile(a, q), 100 - q, lw=1.4, color=cmap[k], label=nm)
ax.set_xscale("log"); ax.set_yscale("log")
ax.axvline(2.0, color="#c8442b", lw=1.0, ls="--")
ax.text(2.1, 50, "2 m/s", color="#c8442b", fontsize=7.5, rotation=90, va="center")
ax.set_xlabel("implied horizontal speed (m/s)"); ax.set_ylabel("% of steps exceeding")
ax.set_title("how fast did particles actually move", loc="left", fontsize=10.5, pad=8)

# where they died
ax = axs[2]
ax.set_facecolor("#e6e1d6")
ax.contour(lon2d[::4, ::4], lat2d[::4, ::4], ocean[::4, ::4].astype(float),
           levels=[0.5], colors="#9a968c", linewidths=0.5)
lost = NLIVE < full
if lost.any():
    li = np.clip(NLIVE[lost] - 1, 0, None)
    ax.scatter(LON[lost][np.arange(lost.sum()), li], LAT[lost][np.arange(lost.sum()), li],
               s=12, c=[cmap[c] for c in CLS[lost]], edgecolor="none", alpha=0.85)
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.set_aspect(1 / np.cos(np.deg2rad(np.nanmean(lat2d))))
ax.set_title(f"last written position of the {int(lost.sum())} lost particles",
             loc="left", fontsize=10.5, pad=8)
for a_ in axs:
    for sp in ("top", "right"): a_.spines[sp].set_visible(False)
fig.suptitle("v8 - raw GLORYS, from_nemo, ScipyParticle, AdvectionRK4_3D + CheckError only",
             x=0.005, ha="left", fontsize=12.5, color=INK, y=0.998)
fig.tight_layout(rect=[0, 0, 1, 0.94])
fig.savefig("v8_boundaries.png", dpi=160, facecolor=SURF)
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11.5, 4.4))
for ax, keys, ttl in ((axs[0], ["surface", "subsurf_1cm", "interior_deep"], "near the surface"),
                      (axs[1], ["floor_face", "partial_gap", "deepest_cell"], "near the seabed")):
    for nm in keys:
        if nm not in NAMES: continue
        k = NAMES.index(nm); s = CLS == k
        zz = np.where(np.isfinite(LAT[s]), Z[s], np.nan)
        ax.plot(tdays, np.nanmedian(zz, axis=0), lw=1.7, color=cmap[k], label=nm)
        ax.fill_between(tdays, np.nanpercentile(zz, 10, axis=0),
                        np.nanpercentile(zz, 90, axis=0), color=cmap[k], alpha=0.12, lw=0)
    ax.invert_yaxis(); ax.legend(frameon=False, fontsize=8)
    ax.set_xlabel("days"); ax.set_ylabel("depth (m)")
    ax.set_title(f"{ttl} - median depth, 10-90% band", loc="left", fontsize=10.5, pad=8)
    for sp in ("top", "right"): ax.spines[sp].set_visible(False)
fig.suptitle("do particles seeded on a boundary stay on it?",
             x=0.005, ha="left", fontsize=12.5, color=INK, y=0.995)
fig.tight_layout(rect=[0, 0, 1, 0.93])
fig.savefig("v8_boundary_depth.png", dpi=160, facecolor=SURF)
plt.show()

## 10. The time boundary

The one boundary the main run cannot reach: `allow_time_extrapolation=False` means a particle
still alive after the last available snapshot must fail with `ErrorTimeExtrapolation`. A short
probe - a handful of particles, deliberately run past the end of the three monthly files -
confirms the boundary exists and is enforced rather than silently extrapolated.

In [ ]:
with xr.open_dataset(GLORYS["W"][-1], decode_times=True) as ds:
    t_last = np.datetime64(ds.time_counter.values[-1], "s")
print(f"last snapshot in the GLORYS files: {t_last}")
over_days = int((t_last - np.datetime64(start_time, "s")) / np.timedelta64(1, "D")) + 6

probe = os.path.join(out_root, "time_boundary_probe.zarr")
if os.path.exists(probe) and not FORCE_RERUN:
    print("reusing", probe)
else:
    if os.path.exists(probe): shutil.rmtree(probe)
    fn = {v: {"data": GLORYS[v], "lon": Hgr, "lat": Hgr, "depth": GLORYS["W"][0]}
          for v in ("U", "V", "W")}
    fs = FieldSet.from_nemo(fn, {"U": "vozocrtx", "V": "vomecrty", "W": "vovecrtz"},
                            {v: {"lon": "glamf", "lat": "gphif", "depth": "depthw",
                                 "time": "time_counter"} for v in ("U", "V", "W")},
                            deferred_load=True, allow_time_extrapolation=False)
    k  = NAMES.index("interior_deep")
    sl = slice(0, 12)
    plo, pla, pdp = (start_lon[start_cls == k][sl], start_lat[start_cls == k][sl],
                     start_dep[start_cls == k][sl])
    ps = ParticleSet(fs, BoundaryParticle, lon=plo, lat=pla, depth=pdp,
                     time=np.datetime64(start_time),
                     cls=np.full(len(plo), k, dtype=np.int32))
    pf = ps.ParticleFile(name=probe, outputdt=timedelta(days=1))
    with quiet():
        ps.execute([AdvectionRK4_3D, CheckError], runtime=timedelta(days=over_days),
                   dt=timedelta(minutes=time_step_minutes), output_file=pf)

dp = xr.open_zarr(probe).compute()
dd = np.nanmax(np.nan_to_num(dp.died.values), axis=1).astype(int)
nl = np.isfinite(dp.lat.values).sum(1)
print(f"\nprobe: {dp.sizes['trajectory']} particles run for {over_days} days "
      f"(last snapshot is at day {int((t_last - np.datetime64(start_time,'s')) / np.timedelta64(1,'D'))})")
print(f"  still alive at the end : {int((nl >= nl.max()).sum())}")
for c in sorted(set(dd.tolist())):
    print(f"  died with {CODES.get(c, c):<24} {int((dd == c).sum())}")
print(f"  last write at day      : {float(np.nanmax(nl)) - 1:.0f}")

## 11. Reading the result

Filled in below against the five predictions in the header.